# Feature Engineering 

Features to create:

1. Lag features (1,3,6)
2. Rolling averages (3,6)
3. Price change features
4. Conflict event flag
5. Risk level label
6. Interaction features
7. Outlier flagging
8. Scaling

In [32]:

#? import libraries

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler #* for normalization
import warnings
warnings.filterwarnings('ignore') #~ to ignore spam warnings on output

In [33]:

#* load dataset
master = pd.read_csv(r'C:/Users/anasq/OneDrive/Desktop/PetroAnalyst/Dataset/ML_Ready_Data/master_dataset.csv')

#^ reset order by dates and drop older index for safety
master = master.sort_values('date').reset_index(drop=True)

#& inspection
print('shape : ', master.shape)
print(list(master.columns))
master.head()

shape :  (195, 8)
['date', 'brent_price', 'wti_price', 'GPR', 'usd_rate', 'avg_lpg_price', 'avg_petrol', 'avg_diesel']


,date,brent_price,wti_price,GPR,usd_rate,avg_lpg_price,avg_petrol,avg_diesel
0,2010/01/01,76.17,78.33,91.58,45.8944,353.4,50.89,40.04
1,2010/02/01,73.75,76.39,80.73,46.2732,353.4,50.35,37.31
2,2010/03/01,78.83,81.20,74.12,45.4509,358.4,50.89,40.04
3,2010/04/01,84.82,84.29,88.76,44.4440,363.4,50.89,40.04
4,2010/05/01,75.95,73.74,88.96,45.7690,368.4,51.44,40.50


# 1. Lag features (monthly)

--> lag'n' = the actual values before n months

In [34]:

#~ crude oil lag
#? takes 1-3 months to reflect lpg/fuel prices

master['brent_lag1'] = master['brent_price'].shift(1)
master['brent_lag3'] = master['brent_price'].shift(3)
master['brent_lag6'] = master['brent_price'].shift(6)

master['wti_lag1'] = master['wti_price'].shift(1)
master['wti_lag3'] = master['wti_price'].shift(3)

#! GPR lag (takes time to hit)
master['GPR_lag1'] = master['GPR'].shift(1)
master['GPR_lag3'] = master['GPR'].shift(3)

#^ USD/INR rate (affect on import and price)
master['usd_rate_lag1'] = master['usd_rate'].shift(1)
master['usd_rate_lag3'] = master['usd_rate'].shift(3)

#todo lpg price lag (future price depends on past price)
master['lpg_price_lag1'] = master['avg_lpg_price'].shift(1)
master['lpg_price_lag3'] = master['avg_lpg_price'].shift(3)

#* fuel price lag (depneds on crude oil import price)
master['petrol_price_lag1'] = master['avg_petrol'].shift(1)
master['petrol_price_lag3'] = master['avg_petrol'].shift(3)

master['diesel_price_lag1'] = master['avg_diesel'].shift(1)
master['diesel_price_lag3'] = master['avg_diesel'].shift(3)

#& inspection
print('Lag features created!\n')
print(master.shape)
print(master.head(3))

Lag features created!

(195, 23)
         date  brent_price  wti_price    GPR  usd_rate  avg_lpg_price  \
0  2010/01/01        76.17      78.33  91.58   45.8944          353.4   
1  2010/02/01        73.75      76.39  80.73   46.2732          353.4   
2  2010/03/01        78.83      81.20  74.12   45.4509          358.4   

   avg_petrol  avg_diesel  brent_lag1  brent_lag3  ...  GPR_lag1  GPR_lag3  \
0       50.89       40.04         NaN         NaN  ...       NaN       NaN   
1       50.35       37.31       76.17         NaN  ...     91.58       NaN   
2       50.89       40.04       73.75         NaN  ...     80.73       NaN   

   usd_rate_lag1  usd_rate_lag3  lpg_price_lag1  lpg_price_lag3  \
0            NaN            NaN             NaN             NaN   
1        45.8944            NaN           353.4             NaN   
2        46.2732            NaN           353.4             NaN   

   petrol_price_lag1  petrol_price_lag3  diesel_price_lag1  diesel_price_lag3  
0           

# 2. Rolling averages

--> rolling'n' = average of last n rows

In [35]:

#todo 3 months rolling averages of external factors (crude oil, GPR & USD rate)
master['brent_roll3'] = master['brent_price'].rolling(window=3).mean().round(4)
master['GPR_roll3'] = master['GPR'].rolling(window=3).mean().round(4)
master['usd_rate_roll3'] = master['usd_rate'].rolling(window=3).mean().round(4)

#^ 6 months rolling averages (brent + GPR)
master['brent_roll6'] = master['brent_price'].rolling(window=6).mean().round(4)
master['GPR_roll6'] = master['GPR'].rolling(window=6).mean().round(4)

print('Rolling averages added!\n')
print('new shape : ', master.shape)
print(master.head(3))

Rolling averages added!

new shape :  (195, 28)
         date  brent_price  wti_price    GPR  usd_rate  avg_lpg_price  \
0  2010/01/01        76.17      78.33  91.58   45.8944          353.4   
1  2010/02/01        73.75      76.39  80.73   46.2732          353.4   
2  2010/03/01        78.83      81.20  74.12   45.4509          358.4   

   avg_petrol  avg_diesel  brent_lag1  brent_lag3  ...  lpg_price_lag3  \
0       50.89       40.04         NaN         NaN  ...             NaN   
1       50.35       37.31       76.17         NaN  ...             NaN   
2       50.89       40.04       73.75         NaN  ...             NaN   

   petrol_price_lag1  petrol_price_lag3  diesel_price_lag1  diesel_price_lag3  \
0                NaN                NaN                NaN                NaN   
1              50.89                NaN              40.04                NaN   
2              50.35                NaN              37.31                NaN   

   brent_roll3  GPR_roll3  usd_rate_r

# 3. percent change feature (monthly)

--> pct_change() = % change in current val compare to previous one

In [36]:

#? percent changes
master['brent_pct_change'] = (master['brent_price'].pct_change() * 100).round(4)
master['wti_pct_change'] = (master['wti_price'].pct_change() * 100).round(4)
master['GPR_pct_change'] = (master['GPR'].pct_change() * 100).round(4)
master['usd_pct_change'] = (master['usd_rate'].pct_change() * 100).round(4)

master['lpg_pct_change'] = (master['avg_lpg_price'].pct_change() * 100).round(4)
master['petrol_pct_change'] = (master['avg_petrol'].pct_change() * 100).round(4)
master['diesel_pct_change'] = (master['avg_diesel'].pct_change() * 100).round(4)

print('percent changes feature added!\n')
print('new shape : ', master.shape)
print(master.head(3))

percent changes feature added!

new shape :  (195, 35)
         date  brent_price  wti_price    GPR  usd_rate  avg_lpg_price  \
0  2010/01/01        76.17      78.33  91.58   45.8944          353.4   
1  2010/02/01        73.75      76.39  80.73   46.2732          353.4   
2  2010/03/01        78.83      81.20  74.12   45.4509          358.4   

   avg_petrol  avg_diesel  brent_lag1  brent_lag3  ...  usd_rate_roll3  \
0       50.89       40.04         NaN         NaN  ...             NaN   
1       50.35       37.31       76.17         NaN  ...             NaN   
2       50.89       40.04       73.75         NaN  ...         45.8728   

   brent_roll6  GPR_roll6  brent_pct_change  wti_pct_change  GPR_pct_change  \
0          NaN        NaN               NaN             NaN             NaN   
1          NaN        NaN           -3.1771         -2.4767        -11.8476   
2          NaN        NaN            6.8881          6.2966         -8.1878   

   usd_pct_change  lpg_pct_change  pet

# 4. Conflict event flag

In [37]:

#! global conflict dates
conflict_dates = {
    '2011/02/01': 'Arab Spring (Libya, Syria, Egypt unrest)',
    '2011/03/01': 'Arab Spring (Libya, Syria, Egypt unrest)',
    '2011/04/01': 'Arab Spring (Libya, Syria, Egypt unrest)',
    '2012/01/01': 'Iran sanctions imposed by US/EU',
    '2012/02/01': 'Iran sanctions imposed by US/EU',
    '2012/07/01': 'Iran sanctions imposed by US/EU',
    '2014/06/01': 'ISIS surge in Iraq',
    '2014/07/01': 'ISIS surge in Iraq',
    '2014/08/01': 'ISIS surge in Iraq',
    '2015/03/01': 'Yemen war start (Houthi-Saudi conflict)',
    '2015/04/01': 'Yemen war start (Houthi-Saudi conflict)',
    '2018/05/01': 'US exits Iran nuclear deal (JCPOA)',
    '2018/06/01': 'US exits Iran nuclear deal (JCPOA)',
    '2019/05/01': 'US-Iran Gulf tensions, tanker attacks',
    '2019/06/01': 'US-Iran Gulf tensions, tanker attacks',
    '2019/09/01': 'US-Iran Gulf tensions, tanker attacks',
    '2020/01/01': 'Soleimani assassination',
    '2022/02/01': 'Russia-Ukraine war',
    '2022/03/01': 'Russia-Ukraine war',
    '2022/04/01': 'Russia-Ukraine war',
    '2023/10/01': 'Israel-Palestine war',
    '2023/11/01': 'Israel-Palestine war',
    '2023/12/01': 'Israel-Palestine war',
    '2024/01/01': 'Houthi Red Sea attacks',
    '2024/02/01': 'Houthi Red Sea attacks',
    '2024/03/01': 'Houthi Red Sea attacks',
    '2024/04/01': 'Iran-Israel direct strikes',
    '2024/05/01': 'Iran-Israel direct strikes',
    '2026/01/01': 'Iran-USA 2026 conflict',
    '2026/02/01': 'Iran-USA 2026 conflict',
    '2026/03/01': 'Iran-USA 2026 conflict'
}

#todo check and flag conflict row in data
master['conflict_event'] = master['date'].isin(conflict_dates).astype(int)
print('conflict rows flagged! total', master['conflict_event'].sum(), 'months')

conflict rows flagged! total 31 months


In [38]:

#& dataset values during conflict months (for verification & debugging)
print('\nconflict months in dataset with parameters')
print(master[master['conflict_event']==1].assign(event=master['date'].map(conflict_dates)) \
      [['date', 'event', 'GPR', 'brent_price', 'avg_lpg_price', 'avg_petrol', 'avg_diesel']].to_string())


conflict months in dataset with parameters
           date                                     event     GPR  brent_price  avg_lpg_price  avg_petrol  avg_diesel
13   2011/02/01  Arab Spring (Libya, Syria, Egypt unrest)   90.01       103.72          453.4       61.66       39.89
14   2011/03/01  Arab Spring (Libya, Syria, Egypt unrest)  136.89       114.64          533.4       61.38       39.89
15   2011/04/01  Arab Spring (Libya, Syria, Egypt unrest)   91.02       123.26          533.4       61.38       39.89
24   2012/01/01           Iran sanctions imposed by US/EU   79.63       110.69          880.4       68.82       47.18
25   2012/02/01           Iran sanctions imposed by US/EU   92.45       119.33          880.4       68.82       47.18
30   2012/07/01           Iran sanctions imposed by US/EU   82.81       102.62          922.9       71.76       49.30
53   2014/06/01                        ISIS surge in Iraq   98.77       111.80         1244.4       74.74       61.73
54   2014/07

In [39]:

#~ % change in values during conflict months
print('\nConflict months data with ' + '%' + ' changes in parameters')
print(master[master['conflict_event'] == 1].assign(event=master['date'].map(conflict_dates)) \
      [['date', 'event', 'GPR_pct_change', 'brent_pct_change', 'usd_pct_change', 'lpg_pct_change', 'petrol_pct_change', 'diesel_pct_change']].to_string())


Conflict months data with % changes in parameters
           date                                     event  GPR_pct_change  brent_pct_change  usd_pct_change  lpg_pct_change  petrol_pct_change  diesel_pct_change
13   2011/02/01  Arab Spring (Libya, Syria, Egypt unrest)         13.3342            7.4596          0.0099          9.6759             0.0000             0.0000
14   2011/03/01  Arab Spring (Libya, Syria, Egypt unrest)         52.0831           10.5283         -1.0251         17.6445            -0.4541             0.0000
15   2011/04/01  Arab Spring (Libya, Syria, Egypt unrest)        -33.5087            7.5192         -1.3655          0.0000             0.0000             0.0000
24   2012/01/01           Iran sanctions imposed by US/EU         -6.6909            2.6143         -2.6362          0.0000             0.0000             0.0000
25   2012/02/01           Iran sanctions imposed by US/EU         16.0995            7.8056         -3.5691          0.0000             0.0

# 5. Risk level label

--> based on GPR index value

In [40]:

#! Risk level function
def assign_risk(GPR):
    if(GPR < 100):
        return 0 # low
    elif(GPR < 150):
        return 1 # medium
    elif(GPR < 200):
        return 2 # high
    else:
        return 3 # critical
    
#? assign risk level to each row
master['risk_level'] = master['GPR'].apply(assign_risk)

#^ risk level distribution
risk_map = {0 : 'low', 1 : 'medium', 2 : 'high', 3 : 'critical'}
dist = master['risk_level'].map(risk_map).value_counts()

print('\n\nRisk distribution\n')
print(dist)

print('\npercentage wise\n')
print((dist / len(master) * 100).round(1))



Risk distribution

risk_level
low         107
medium       74
high         10
critical      4
Name: count, dtype: int64

percentage wise

risk_level
low         54.9
medium      37.9
high         5.1
critical     2.1
Name: count, dtype: float64


# 6. Interaction features

--> combine multiple features to give model better insights

In [41]:

#? 1. crude pil cost in INR / barrel
master['crude_inr_impact'] = (master['brent_price'] * master['usd_rate']).round(4)

#^ 2. geopolitical impact on crude oil price
master['conflict_crude_index'] = ((master['GPR'] / 100) * master['brent_price']).round(4)

#* 3. brent-WTI spread (high GPR -> wide spread)
master['brent_wti_spread'] = (master['brent_price'] - master['wti_price']).round(4)

#! 4. INR stress during conflicts (how high GPR affects INR)
master['inr_gpr_stress'] = (master['usd_rate'] * (master['GPR'] / 100)).round(4)

#~ 5. crude lpg impact (how crude is driving lpg price?)
master['crude_lpg_impact'] = ((master['brent_price'] * master['usd_rate']) / master['avg_lpg_price']).round(4)

#todo 6. LPG price momentum - is LPG falling or rising in last 3 months?
master['lpg_momentum'] = (master['avg_lpg_price'].diff(3) / 3).round(4)

#& 7. conflict intensity - combine GPR + crude spike (z-normalized)
master['conflict_intensity'] = (
    (master['GPR'] - master['GPR'].mean()) / master["GPR"].std() +
    (master['brent_price'] - master['brent_price'].mean()) / master['brent_price']
    ).round(4)

print('interaction features added!')
print('new shape : ', master.shape)
master.head()

interaction features added!
new shape :  (195, 44)


,date,brent_price,wti_price,GPR,usd_rate,avg_lpg_price,avg_petrol,avg_diesel,brent_lag1,brent_lag3,...,diesel_pct_change,conflict_event,risk_level,crude_inr_impact,conflict_crude_index,brent_wti_spread,inr_gpr_stress,crude_lpg_impact,lpg_momentum,conflict_intensity
0,2010/01/01,76.17,78.33,91.58,45.8944,353.4,50.89,40.04,NaN,NaN,...,NaN,0,0,3495.7764,69.7565,-2.16,42.0301,9.8918,NaN,-0.3991
1,2010/02/01,73.75,76.39,80.73,46.2732,353.4,50.35,37.31,76.17,NaN,...,-6.8182,0,0,3412.6485,59.5384,-2.64,37.3564,9.6566,NaN,-0.7379
2,2010/03/01,78.83,81.20,74.12,45.4509,358.4,50.89,40.04,73.75,NaN,...,7.3171,0,0,3582.8944,58.4288,-2.37,33.6882,9.9969,NaN,-0.8561
3,2010/04/01,84.82,84.29,88.76,44.4440,363.4,50.89,40.04,78.83,76.17,...,0.0000,0,0,3769.7401,75.2862,0.53,39.4485,10.3735,3.3333,-0.3745
4,2010/05/01,75.95,73.74,88.96,45.7690,368.4,51.44,40.50,84.82,73.75,...,1.1489,0,0,3476.1556,67.5651,2.21,40.7161,9.4358,5.0000,-0.4758


# 7. Outlier flagging

In [42]:

#* flag outliers 

#^ based on quartile values (GPR only)
def flag_outlier_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.0 * IQR
    upper = Q3 + 1.0 * IQR

    return ((series < lower) | (series > upper)).astype(int)

#^ based on % changes (for other colms)
def flag_outlier_pct(series, threshold=8.0):
    pct_change = series.pct_change() * 100

    return (pct_change.abs() > threshold).astype(int)

#todo apply to features (with different threasholds)
master['GPR_outlier'] = flag_outlier_iqr(master['GPR'])
master['brent_outlier'] = flag_outlier_pct(master['brent_price'], 18.0)
master['lpg_price_outlier'] = flag_outlier_pct(master['avg_lpg_price'], 10.0)
master['petrol_price_outlier'] = flag_outlier_pct(master['avg_petrol'], 5.0)
master['diesel_price_outlier'] = flag_outlier_pct(master['avg_diesel'])
master['usd_outliers'] = flag_outlier_pct(master['usd_rate'], 3.0)

print('outliers flagged! (1 : outlier, 0 : normal)')

#& inspection
print('GPR outliers : ', master['GPR_outlier'].sum())
print('brent outliers : ', master['brent_outlier'].sum())
print('lpg price outliers : ', master['lpg_price_outlier'].sum())
print('petrol price outlier : ', master['petrol_price_outlier'].sum())
print('diesel price outlier : ', master['diesel_price_outlier'].sum())
print('usd rate outliers : ', master['usd_outliers'].sum())

print('outlier months : \n')
print(master[master['GPR_outlier'] == 1][['date','GPR','brent_price','avg_lpg_price','avg_petrol','avg_diesel','usd_rate']])

outliers flagged! (1 : outlier, 0 : normal)
GPR outliers :  12
brent outliers :  13
lpg price outliers :  14
petrol price outlier :  14
diesel price outlier :  7
usd rate outliers :  14
outlier months : 

           date     GPR  brent_price  avg_lpg_price  avg_petrol  avg_diesel  \
145  2022/02/01  216.16        97.13          902.9      100.26       88.04   
146  2022/03/01  318.95       117.25          952.9      100.26       88.04   
147  2022/04/01  191.14       104.58         1002.9      100.26       88.04   
165  2023/10/01  197.89        90.60          911.7      103.26       91.04   
166  2023/11/01  156.70        82.94          911.7      103.26       91.04   
168  2024/01/01  160.37        80.12          911.7      105.26       92.04   
171  2024/04/01  163.95        89.94          811.7      105.26       92.04   
182  2025/03/01  173.91        72.73          811.7      110.26       97.04   
184  2025/05/01  165.66        64.45          861.7      110.26       97.04   
185  

# 8. Rename columns
--> change colm names for better understanding

In [48]:
master = master.rename(columns={'GPR': 'GPR_index', 'avg_lpg_price': 'LPG_price', 'avg_petrol': 'petrol_price', 'avg_diesel': 'diesel_price', 'usd_rate': 'USD/INR_rate'})

# 8. clean data after adding new features 
--> drop first 6 rows (which contains missing values) 

--> standerize feature names

In [43]:

#? remove top 6 rows (contains missing values)
print('shape before cleaning : ', master.shape)
print('missing values : ', master.isnull().sum().sum())

master = master.dropna().reset_index(drop=True)

print('shape after cleaning : ', master.shape)
print('missing values after cleaning : ', master.isnull().sum().sum())

#~ round all colms value to 4 decimal points
numeric_cols = master.select_dtypes(include='number').columns
master[numeric_cols] = master[numeric_cols].round(4)

shape before cleaning :  (195, 50)
missing values :  60
shape after cleaning :  (189, 50)
missing values after cleaning :  0


In [49]:

#! dataset preview
master.head(10)

,date,brent_price,wti_price,GPR_index,USD/INR_rate,LPG_price,petrol_price,diesel_price,brent_lag1,brent_lag3,...,inr_gpr_stress,crude_lpg_impact,lpg_momentum,conflict_intensity,GPR_outlier,brent_outlier,lpg_price_outlier,petrol_price_outlier,diesel_price_outlier,usd_outliers
0,2010/07/01,75.58,76.32,79.38,46.7617,378.4,54.45,42.32,74.76,84.82,...,37.1194,9.3400,5.0000,-0.7504,0,0,0,0,0,0
1,2010/08/01,77.04,76.60,80.99,46.4605,383.4,54.55,39.88,75.58,75.95,...,37.6284,9.3357,5.0000,-0.6856,0,0,0,0,0,0
2,2010/09/01,77.84,75.24,71.17,45.8729,388.4,54.77,39.98,77.04,74.76,...,32.6477,9.1935,5.0000,-0.9516,0,0,0,0,0,0
3,2010/10/01,82.67,81.89,65.99,44.3540,393.4,55.47,39.89,77.84,75.58,...,29.2692,9.3207,5.0000,-1.0392,0,0,0,0,0,1
4,2010/11/01,85.28,84.25,94.71,44.9315,398.4,55.78,39.89,82.67,77.04,...,42.5546,9.6179,5.0000,-0.2021,0,0,0,0,0,0
5,2010/12/01,91.45,89.15,97.20,45.1000,403.4,59.08,39.89,85.28,77.84,...,43.8372,10.2241,5.0000,-0.0707,0,0,0,1,0,0
6,2011/01/01,96.52,89.17,79.42,45.3750,413.4,61.66,39.89,91.45,82.67,...,36.0368,10.5941,6.6667,-0.5265,0,0,0,0,0,0
7,2011/02/01,103.72,88.58,90.01,45.3795,453.4,61.66,39.89,96.52,85.28,...,40.8461,10.3810,18.3333,-0.1726,0,0,0,0,0,0
8,2011/03/01,114.64,102.86,136.89,44.9143,533.4,61.38,39.89,103.72,91.45,...,61.4832,9.6531,43.3333,1.2180,0,0,1,0,0,0
9,2011/04/01,123.26,109.53,91.02,44.3010,533.4,61.38,39.89,114.64,96.52,...,40.3228,10.2372,40.0000,-0.0256,0,0,0,0,0,0


In [50]:
master.reset_index(drop=True)

,date,brent_price,wti_price,GPR_index,USD/INR_rate,LPG_price,petrol_price,diesel_price,brent_lag1,brent_lag3,...,inr_gpr_stress,crude_lpg_impact,lpg_momentum,conflict_intensity,GPR_outlier,brent_outlier,lpg_price_outlier,petrol_price_outlier,diesel_price_outlier,usd_outliers
0,2010/07/01,75.58,76.32,79.38,46.7617,378.4,54.45,42.32,74.76,84.82,...,37.1194,9.3400,5.0,-0.7504,0,0,0,0,0,0
1,2010/08/01,77.04,76.60,80.99,46.4605,383.4,54.55,39.88,75.58,75.95,...,37.6284,9.3357,5.0,-0.6856,0,0,0,0,0,0
2,2010/09/01,77.84,75.24,71.17,45.8729,388.4,54.77,39.98,77.04,74.76,...,32.6477,9.1935,5.0,-0.9516,0,0,0,0,0,0
3,2010/10/01,82.67,81.89,65.99,44.3540,393.4,55.47,39.89,77.84,75.58,...,29.2692,9.3207,5.0,-1.0392,0,0,0,0,0,1
4,2010/11/01,85.28,84.25,94.71,44.9315,398.4,55.78,39.89,82.67,77.04,...,42.5546,9.6179,5.0,-0.2021,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184,2025/11/01,63.80,60.06,104.34,88.8444,861.7,110.26,97.04,64.54,67.87,...,92.7002,6.5780,0.0,-0.2375,0,0,0,0,0,0
185,2025/12/01,62.54,57.97,131.39,90.0350,861.7,110.26,97.04,63.80,67.99,...,118.2970,6.5345,0.0,0.4993,0,0,0,0,0,0
186,2026/01/01,66.60,60.04,167.80,90.9025,861.7,107.26,94.04,62.54,64.54,...,152.5344,7.0258,0.0,1.5997,1,0,0,0,0,0
187,2026/02/01,70.89,64.51,121.62,90.7458,861.7,107.26,94.04,66.60,63.80,...,110.3650,7.4654,0.0,0.3705,0,0,0,0,0,0


In [ ]:
master.shape

(189, 50)

# 9. Create scaled version for Machine Learning Models
--> normalize values between 0-1 for ML algorithms

In [54]:

#* normalize numeric colms

#^ colms to not scale
dont_scale = [
    'date',
    'risk_level',
    'conflict_event',
    'brent_outlier',
    'GPR_outlier',
    'lpg_outlier',
    'usd_outlier',
    'petrol_outlier',
    'diesel_outlier'
]

#todo list comprehension to scale colms
scale_cols = [col for col in master.columns if col not in dont_scale]

#& create a scaler
scaler = MinMaxScaler()
master_scaled = master.copy() #? copy dataframe
master_scaled[scale_cols] = scaler.fit_transform(master[scale_cols])

print('dataframe scaled!')
print('DataFrame preview :\n')
master_scaled.head()

dataframe scaled!
DataFrame preview :



,date,brent_price,wti_price,GPR_index,USD/INR_rate,LPG_price,petrol_price,diesel_price,brent_lag1,brent_lag3,...,inr_gpr_stress,crude_lpg_impact,lpg_momentum,conflict_intensity,GPR_outlier,brent_outlier,lpg_price_outlier,petrol_price_outlier,diesel_price_outlier,usd_outliers
0,2010/07/01,0.534230,0.608098,0.080451,0.050718,0.000000,0.000000,0.042687,0.526571,0.620529,...,0.031827,0.847072,0.634667,0.328556,0,0,0.0,0.0,0.0,0.0
1,2010/08/01,0.547866,0.610947,0.086631,0.044510,0.005774,0.001792,0.000000,0.534230,0.537686,...,0.033891,0.846548,0.634667,0.334678,0,0,0.0,0.0,0.0,0.0
2,2010/09/01,0.555338,0.597111,0.048939,0.032399,0.011547,0.005734,0.001749,0.547866,0.526571,...,0.013697,0.829208,0.634667,0.309549,0,0,0.0,0.0,0.0,0.0
3,2010/10/01,0.600448,0.664768,0.029056,0.001092,0.017321,0.018276,0.000175,0.555338,0.534230,...,0.000000,0.844719,0.634667,0.301273,0,0,0.0,0.0,0.0,1.0
4,2010/11/01,0.624825,0.688778,0.139293,0.012995,0.023095,0.023831,0.000175,0.600448,0.547866,...,0.053863,0.880960,0.634667,0.380354,0,0,0.0,0.0,0.0,0.0


# 10. Final summary

In [ ]:
print('='*60)
print('Feature Engineering Completed!')
print('='*60)

print(f'Total rows : {master.shape[0]}')
print(f'Total columns : {master.shape[1]}')
print(f'Date ranges : {master['date'].iloc[0]} --> {master['date'].iloc[-1]}')
print(f'NULL values {master.isnull().sum().sum()}')
print(f'Total conflict months : {master['conflict_event'].sum()}')
print(f'\nrisk level distribution : ')
risk_map = {0:"Low", 1:"Medium", 2:"High", 3:"Critical"}
print(master['risk_level'].map(risk_map).value_counts())

print('\nAll columns : \n')
for i, col in enumerate(master.columns, 1):
    print(f'  {i:2}. {col}')

Feature Engineering Completed!
Total rows : 189
Total columns : 50
Date ranges : 2010/07/01 --> 2026/03/01
NULL values 0
Total conflict months : 31

risk level distribution : 
risk_level
Low         101
Medium       74
High         10
Critical      4
Name: count, dtype: int64

All columns : 

   1. date
   2. brent_price
   3. wti_price
   4. GPR_index
   5. USD/INR_rate
   6. LPG_price
   7. petrol_price
   8. diesel_price
   9. brent_lag1
  10. brent_lag3
  11. brent_lag6
  12. wti_lag1
  13. wti_lag3
  14. GPR_lag1
  15. GPR_lag3
  16. usd_rate_lag1
  17. usd_rate_lag3
  18. lpg_price_lag1
  19. lpg_price_lag3
  20. petrol_price_lag1
  21. petrol_price_lag3
  22. diesel_price_lag1
  23. diesel_price_lag3
  24. brent_roll3
  25. GPR_roll3
  26. usd_rate_roll3
  27. brent_roll6
  28. GPR_roll6
  29. brent_pct_change
  30. wti_pct_change
  31. GPR_pct_change
  32. usd_pct_change
  33. lpg_pct_change
  34. petrol_pct_change
  35. diesel_pct_change
  36. conflict_event
  37. risk_level
 

# 11. Save both files

In [56]:
master.to_csv('EDA_data.csv', index=False)
master_scaled.to_csv('ML_ready_data.csv', index=False)

print('\nFiles saved:')
print('  EDA_data.csv  -> use for EDA & visualization')
print('  ML_ready_data.csv    -> use for ML algorithms')


Files saved:
  EDA_data.csv  -> use for EDA & visualization
  ML_ready_data.csv    -> use for ML algorithms
